<a id='home'></a>
### purpose

estimate fitness of individuals in the common garden on the spatially continuous simulation landscape

### outline
1. [get environmental data for each garden](#envdata)
1. [estimate fitness in each common garden](#fit)

In [1]:
from pythonimports import *

import runtime_API as rt  # contains fitness_calculator.R

outerdir = '/work/lotterhos/brandon/continuous_space_runtime'

# continuous space sims directory
tdir = '/home/b.lind/offsets/run_20220919_tutorial/tutorial'

# directories with files
training_dir = f'{outerdir}/gradient_forests/training/training_files'
fitting_dir = f'{outerdir}/gradient_forests/fitting'
garden_dir = f'{fitting_dir}/garden_files'

# where to save files
fitness_dir = makedir(f'{outerdir}/fitness_mats')

# parallel engines
lview, dview = get_client(cluster_id='1733328168-jy14', profile='lotterhos')

t0 = dt.now()  # notebook timer

rt.latest_commit()
session_info.show()

36 36
#########################################################
Today:	December 04, 2024 - 12:40:37 EST
python version: 3.8.5
conda env: mvp_env

Current commit of pythonimports:
commit 6a767410e7b569adbf9df526de108f22ef50aad8  
Author: Brandon Lind <lind.brandon.m@gmail.com>  
Date:   Wed Mar 6 13:42:13 2024 -0700

Current commit of MVP_offsets:
commit 5ce82f4d655645237a0f4026fa32e220226dc373  
Author: Brandon Lind <lind.brandon.m@gmail.com>  
Date:   Thu May 16 13:02:58 2024 -0400

Current commit of MVP_runtime:
commit a98ef38f88b81d808c334e87d0c6392e142941da  
Merge: 50cf2c9 cc54cc3  
Author: Brandon Lind <lind.brandon.m@gmail.com>
#########################################################



<a id='envdata'></a>
# get environmental data for each garden

[top](#home)

In [2]:
garden_files = fs(garden_dir)

print(len(garden_files))

garden_files[0]

100


'/work/lotterhos/brandon/continuous_space_runtime/gradient_forests/fitting/garden_files/tutorial_GF_gardenfile_1.txt'

In [3]:
# note each column has one value, so I can use the first row when calculating fitness to a specific garden
    # see calc_R_fitness calls
pd.read_table(garden_files[0], index_col=0)

,env1_mat,env2_MTWetQ,env3_MTDQ,env4_PDM,env5_PwarmQ,env6_PWM
25_33,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
85_34,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
57_44,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
31_45,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
11_46,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
...,...,...,...,...,...,...
22_8575,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
75_8580,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
62_8582,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569
79_8585,0.186225,-0.226935,0.861531,0.221974,-0.363974,0.761569


In [4]:
garden_data = {}
for f in pbar(garden_files):
    garden_ID = op.basename(f).split('.')[0].split("_")[-1]
    garden_data[garden_ID] = pd.read_table(f, index_col=0)
    
garden_data[garden_ID]

100%|███████████████| 100/100 [00:00<00:00, 226.52it/s]


,env1_mat,env2_MTWetQ,env3_MTDQ,env4_PDM,env5_PwarmQ,env6_PWM
25_33,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
85_34,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
57_44,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
31_45,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
11_46,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
...,...,...,...,...,...,...
22_8575,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
75_8580,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
62_8582,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201
79_8585,0.034177,0.399089,-0.440848,-0.464035,0.673533,0.360201


In [5]:
garden_data[garden_ID].iloc[0]

env1_mat       0.034177
env2_MTWetQ    0.399089
env3_MTDQ     -0.440848
env4_PDM      -0.464035
env5_PwarmQ    0.673533
env6_PWM       0.360201
Name: 25_33, dtype: float64

<a id='fit'></a>
# estimate fitness in each common garden

[top](#home)

In [6]:
# look at phenodata read in by fitness_calculator.R
phenodata = pd.read_table(
    '/work/lotterhos/brandon/continuous_space_runtime/individual_data.txt',
)

traits = ["phenotype1_mat",  # list copied from fitness_calculator.R
           "phenotype2_MTWetQ",
           "phenotype3_MTDQ",
           "phenotype4_PDM",
           "phenotype5_PwarmQ",
           "phenotype6_PWM"]

phenodata[traits]

,phenotype1_mat,phenotype2_MTWetQ,phenotype3_MTDQ,phenotype4_PDM,phenotype5_PwarmQ,phenotype6_PWM
0,0.737967,-0.191421,0.142992,-0.131894,-0.452497,-0.541651
1,-0.274081,-0.428332,-0.012880,0.221058,0.229296,0.066597
2,-0.676326,0.086629,-0.470671,0.218813,0.260098,-0.103465
3,-0.283221,-0.403942,-0.333718,-0.200554,-0.345915,-0.410001
4,0.057616,-0.303201,0.731600,0.195679,-0.367110,0.196296
...,...,...,...,...,...,...
995,-0.070241,-0.367056,-0.061876,-0.308041,-0.468111,-0.508108
996,-0.050435,-0.342693,0.128758,0.107181,-0.061796,-0.081682
997,0.354088,0.493751,0.064744,-0.243331,-0.164916,-0.244379
998,-0.009027,0.424444,-0.575177,-0.514613,0.586778,0.266235


In [7]:
def calc_R_fitness(*args):
    import subprocess
    
    args = [str(arg) for arg in args]
    
    Rscript = '/home/b.lind/anaconda3/envs/MVP_env_R4.0.3/bin/Rscript'
    script_file = '/home/b.lind/code/06_run_time_project/07_continuous_space_sims/fitness_calculator.R'
    
    output = subprocess.check_output(
        [
            Rscript,
            script_file,
            *args
        ]
    )

    return output

In [8]:
fitness_dir

'/work/lotterhos/brandon/continuous_space_runtime/fitness_mats'

In [9]:
jobs = []

for gardenID, data in garden_data.items():
    (opt0, opt1, opt2, opt3, opt4, opt5) = data.iloc[0]

    output_file = op.join(fitness_dir, f'tutorial_{gardenID}.txt')

    jobs.append(
        lview.apply_async(
            calc_R_fitness, *('tutorial',
                              output_file, 
                              opt0, opt1, opt2, opt3, opt4, opt5)
        )
    )

watch_async(jobs)


Watching 100 jobs ...


100%|███████████████| 100/100 [00:04<00:00, 22.46it/s]


In [10]:
fitness_dir

'/work/lotterhos/brandon/continuous_space_runtime/fitness_mats'

In [11]:
print(jobs[0].r.decode('utf-8'))

R version 4.0.3 (2020-10-10)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: CentOS Linux 7 (Core)

Matrix products: default
BLAS/LAPACK: /work/lotterhos/brandon/anaconda3/envs/MVP_env_R4.0.3/lib/libopenblasp-r0.3.17.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] progress_1.2.2 mvtnorm_1.1-3 

loaded via a namespace (and not attached):
 [1] compiler_4.0.3    ellipsis_0.3.2    R6_2.5.1          cli_3.3.0        
 [5] hms_1.1.1         prettyunits_1.1.1 crayon_1.5.1      vctrs_0.4.1      
 [9] lifecycle_1.0.1   pkgconfig_2.0.3   rlang_1.0.

In [12]:
pd.read_table('/work/lotterhos/brandon/continuous_space_runtime/fitness_mats/tutorial_1.txt')

,25_33,85_34,57_44,31_45,11_46,22_64,47_65,10_72,20_85,64_106,100_119,76_127,11_132,23_164,18_173,29_175,49_194,67_198,6_207,61_215,22_216,47_221,29_224,22_230,9_236,67_254,52_255,64_261,30_268,42_269,1_271,96_280,84_293,19_306,78_309,77_314,34_338,92_348,66_352,63_359,66_382,79_385,40_386,51_397,50_404,36_405,15_412,16_423,26_437,54_438,...,83_8163,7_8169,1_8172,75_8187,33_8217,43_8222,95_8228,40_8229,23_8231,23_8242,97_8252,76_8286,15_8311,64_8326,91_8330,79_8346,12_8351,93_8359,39_8360,62_8367,73_8371,24_8392,98_8394,39_8402,84_8409,75_8417,97_8418,59_8432,43_8446,3_8456,47_8458,99_8462,37_8468,60_8484,58_8500,32_8503,39_8509,45_8516,48_8533,57_8534,55_8535,63_8536,26_8547,45_8559,68_8564,22_8575,75_8580,62_8582,79_8585,78_8590
1,0.51512,0.629394,0.391132,0.445775,0.914052,0.653387,0.646495,0.6926,0.541038,0.754496,0.339901,0.444621,0.695847,0.519039,0.828569,0.493857,0.46052,0.686715,0.645799,0.49265,0.655309,0.773413,0.530306,0.655309,0.828569,0.46634,0.4805,0.765004,0.508249,0.41053,0.988822,0.545878,0.649216,0.474823,0.394692,0.431491,0.541283,0.578675,0.632584,0.69827,0.632584,0.478368,0.472253,0.436936,0.472253,0.820122,0.570782,0.799936,0.80355,0.665105,...,0.642307,0.845942,0.974128,0.615396,0.471031,0.499594,0.707139,0.666928,0.496706,0.477771,0.459048,0.630283,0.642003,0.761429,0.552331,0.402566,0.811028,0.641666,0.46052,0.584532,0.705264,0.383429,0.485955,0.438077,0.649216,0.615396,0.451184,0.356118,0.534473,0.809931,0.867853,0.472843,0.589207,0.520398,0.391132,0.521896,0.493857,0.613319,0.775248,0.535861,0.77843,0.784841,0.842294,0.746161,0.391132,0.491407,0.700836,0.541927,0.348332,0.413359


In [13]:
len(fs(fitness_dir))

100

In [14]:
f = fs(fitness_dir)[0]

!ls -l $f

-rw-r--r-- 1 b.lind users 27719 Dec  4 12:34 /work/lotterhos/brandon/continuous_space_runtime/fitness_mats/tutorial_1.txt


In [15]:
!date

Wed Dec  4 12:40:44 EST 2024


In [16]:
formatclock(dt.now() - t0)

'0-00:00:07'